# add commit description

In [ ]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

# Connect to the database
conn = sqlite3.connect('/home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite')

# Read the data
df = pd.read_sql_query("SELECT COMMIT_HASH, DESCRIPTION_IN_PATCH FROM vulnerabilities", conn)

# Function to get commit message from GitHub
def get_commit_message(commit_hash):
    url = f"https://github.com/torvalds/linux/commit/{commit_hash}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find commit message (usually in a div with class containing 'commit-message')
        commit_message_elem = soup.find('div', class_=lambda x: x and 'commit-message' in x)
        if commit_message_elem:
            return commit_message_elem.get_text(strip=True)
        else:
            # Alternative selector
            commit_message_elem = soup.find('p', class_='commit-title')
            if commit_message_elem:
                return commit_message_elem.get_text(strip=True)
    except Exception as e:
        print(f"Error fetching commit {commit_hash}: {e}")
        return None
    return None

# Update empty DESCRIPTION_IN_PATCH fields
for index, row in df.iterrows():
    if pd.isna(row['DESCRIPTION_IN_PATCH']) or row['DESCRIPTION_IN_PATCH'].strip() == '':
        print(f"Fetching commit message for {row['COMMIT_HASH']}")
        commit_message = get_commit_message(row['COMMIT_HASH'])
        if commit_message:
            # Update the database
            cursor = conn.cursor()
            cursor.execute("UPDATE vulnerabilities SET DESCRIPTION_IN_PATCH = ? WHERE COMMIT_HASH = ?", 
                          (commit_message, row['COMMIT_HASH']))
            conn.commit()
            print(f"Updated commit {row['COMMIT_HASH']}")
        
        # Add delay to avoid rate limiting
        time.sleep(1)

conn.close()
print("Database update completed")

# remove comments in the code

In [ ]:
import re
import sqlite3
import pandas as pd

# Connect to both databases
conn1 = sqlite3.connect('/home/azibaeir/Research/VulnLLMEval-SANER/data/database.sqlite')
conn2 = sqlite3.connect('/home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite')

def remove_comments(code, record_id=None, block_type=None):
    if not code:
        return code
    
    original_code = code
    removed_comments = []
    
    # First handle multi-line comments (/* */) across the entire text
    def multi_comment_replacer(match):
        comment = match.group(0)
        removed_comments.append(f"Multi-line comment: {comment}")
        return ''
    
    code = re.sub(r'/\*.*?\*/', multi_comment_replacer, code, flags=re.DOTALL)
    
    lines = code.split('\n')
    cleaned_lines = []
    
    for line_num, line in enumerate(lines, 1):
        # Preserve lines starting with "// File path:"
        if line.strip().startswith('// File path:'):
            cleaned_lines.append(line)
        else:
            # Handle single-line comments more carefully
            in_string = False
            i = 0
            result = ""
            
            while i < len(line):
                if line[i] == '"' and (i == 0 or line[i-1] != '\\'):
                    in_string = not in_string
                    result += line[i]
                elif line[i:i+2] == '//' and not in_string:
                    # Found comment outside of string, capture and log it
                    comment = line[i:]
                    removed_comments.append(f"Single-line comment (line {line_num}): {comment}")
                    break
                else:
                    result += line[i]
                i += 1
            
            # Remove trailing whitespace
            cleaned_lines.append(result.rstrip())
    
    # Log removed comments if any were found
    if removed_comments and record_id is not None:
        print(f"\n=== Comments removed from {block_type} (Record ID: {record_id}) ===")
        for comment in removed_comments:
            print(f"  {comment}")
        print("=" * 60)
    
    return '\n'.join(cleaned_lines)

# Process database.sqlite
print("Processing database.sqlite...")
df1 = pd.read_sql_query("SELECT id, VULNERABLE_CODE_BLOCK, PATCHED_CODE_BLOCK FROM vulnerabilities", conn1)
cursor1 = conn1.cursor()

for index, row in df1.iterrows():
    print(f"\nProcessing record {row['id']} from database.sqlite...")
    
    vulnerable_cleaned = remove_comments(row['VULNERABLE_CODE_BLOCK'], 
                                       record_id=row['id'], 
                                       block_type="VULNERABLE_CODE_BLOCK")
    
    patched_cleaned = remove_comments(row['PATCHED_CODE_BLOCK'], 
                                    record_id=row['id'], 
                                    block_type="PATCHED_CODE_BLOCK")
    
    cursor1.execute("UPDATE vulnerabilities SET VULNERABLE_CODE_BLOCK = ?, PATCHED_CODE_BLOCK = ? WHERE id = ?", 
                   (vulnerable_cleaned, patched_cleaned, row['id']))

conn1.commit()

# Process database_leakagefree.sqlite
print("\n\nProcessing database_leakagefree.sqlite...")
df2 = pd.read_sql_query("SELECT id, VULNERABLE_CODE_BLOCK, PATCHED_CODE_BLOCK FROM vulnerabilities", conn2)
cursor2 = conn2.cursor()

for index, row in df2.iterrows():
    print(f"\nProcessing record {row['id']} from database_leakagefree.sqlite...")
    
    vulnerable_cleaned = remove_comments(row['VULNERABLE_CODE_BLOCK'], 
                                       record_id=row['id'], 
                                       block_type="VULNERABLE_CODE_BLOCK")
    
    patched_cleaned = remove_comments(row['PATCHED_CODE_BLOCK'], 
                                    record_id=row['id'], 
                                    block_type="PATCHED_CODE_BLOCK")
    
    cursor2.execute("UPDATE vulnerabilities SET VULNERABLE_CODE_BLOCK = ?, PATCHED_CODE_BLOCK = ? WHERE id = ?", 
                   (vulnerable_cleaned, patched_cleaned, row['id']))

conn2.commit()

# Close connections
conn1.close()
conn2.close()

print("\n\nComments removed from both databases")

# number of files/functions changed

In [4]:
import re
import sqlite3
import pandas as pd

# Connect to both databases
conn1 = sqlite3.connect('/home/azibaeir/Research/VulnLLMEval-SANER/data/database.sqlite')
conn2 = sqlite3.connect('/home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite')

def count_file_paths(code):
    """Count the number of '// File path:' lines in the code"""
    if not code:
        return 0
    return len(re.findall(r'//\s*File\s+path:', code, re.IGNORECASE))

def count_functions(code):
    """Count the number of function definitions in C code"""
    if not code:
        return 0
    
    function_count = 0
    lines = code.split('\n')
    
    # Clean the code - remove comments but preserve structure
    cleaned_lines = []
    for line in lines:
        # Preserve // File path: lines
        if line.strip().startswith('// File path:'):
            cleaned_lines.append(line)
            continue
            
        # Remove single-line comments
        comment_pos = line.find('//')
        if comment_pos != -1:
            # Check if // is inside a string
            in_string = False
            for i in range(comment_pos):
                if line[i] == '"' and (i == 0 or line[i-1] != '\\'):
                    in_string = not in_string
            if not in_string:
                line = line[:comment_pos]
        
        # Remove multi-line comment markers
        line = re.sub(r'/\*.*?\*/', '', line, flags=re.DOTALL)
        cleaned_lines.append(line.rstrip())
    
    # Look for function definitions by finding opening braces and working backwards
    for i, line in enumerate(cleaned_lines):
        stripped_line = line.strip()
        
        # Look for standalone opening brace (function body start)
        if stripped_line == '{':
            # Work backwards to find the function signature
            signature_lines = []
            j = i - 1
            
            # Collect lines backwards until we find the start of the function
            while j >= 0:
                current_line = cleaned_lines[j].strip()
                
                # Skip empty lines
                if not current_line:
                    j -= 1
                    continue
                
                # Skip preprocessor directives and comments
                if (current_line.startswith('#') or 
                    current_line.startswith('//') or 
                    current_line.startswith('/*') or
                    current_line.startswith('*')):
                    j -= 1
                    continue
                
                # Add this line to the signature
                signature_lines.insert(0, current_line)
                
                # Check if we have a complete function signature
                full_signature = ' '.join(signature_lines)
                
                # Must contain parentheses for parameters
                if '(' in full_signature and ')' in full_signature:
                    # Check if this looks like a function definition
                    if is_function_signature(full_signature, cleaned_lines, j):
                        func_name = extract_function_name_from_signature(full_signature)
                        if func_name:
                            function_count += 1
                            print(f"  Found function definition ending at line {i + 1}: {func_name}")
                        break
                
                # Stop if we hit something that can't be part of a function signature
                if any(keyword in current_line.lower() for keyword in 
                       ['if ', 'else', 'while ', 'for ', 'switch ', 'case ', 'default',
                        'return ', 'break', 'continue', 'goto ', '}', ';']):
                    break
                
                j -= 1
    
    return function_count

def is_function_signature(signature, all_lines, start_line):
    """Determine if a signature is actually a function definition"""
    
    # Must have parentheses
    if '(' not in signature or ')' not in signature:
        return False
    
    # Skip if it contains control flow keywords
    if any(keyword in signature.lower() for keyword in 
           ['if ', 'else', 'while ', 'for ', 'switch ', 'case ', 'default',
            'return ', 'break', 'continue', 'goto ', 'sizeof', 'typeof']):
        return False
    
    # Skip if it contains operators that suggest it's not a function definition
    if any(op in signature for op in ['==', '!=', '<=', '>=', '&&', '||', '!', '+', '-', '*/', '%']):
        return False
    
    # Skip assignments
    if '=' in signature.split('(')[0]:
        return False
    
    # Extract the part before parentheses
    before_paren = signature.split('(')[0].strip()
    if not before_paren:
        return False
    
    # Split into words and find the function name
    words = before_paren.split()
    if not words:
        return False
    
    # Find the last word that could be a function name
    function_name = None
    for word in reversed(words):
        # Clean the word of pointers
        clean_word = word.replace('*', '').strip()
        
        # Skip type specifiers and keywords
        if clean_word.lower() in ['static', 'inline', 'const', 'volatile', 'extern', 
                                 'struct', 'union', 'enum', 'unsigned', 'signed',
                                 'long', 'short', 'int', 'char', 'float', 'double', 
                                 'void', 'size_t', 'ssize_t', 'off_t', 'time_t', 
                                 'pid_t', 'uid_t', 'gid_t', 'dev_t', 'ino_t', 
                                 'mode_t', 'nlink_t', 'blkcnt_t', 'blksize_t',
                                 '__attribute__', '__restrict__', 'register',
                                 'auto', 'restrict', '__inline__']:
            continue
        
        # Must be a valid identifier
        if clean_word and clean_word.replace('_', '').isalnum() and not clean_word[0].isdigit():
            function_name = clean_word
            break
    
    # Must have found a valid function name
    if not function_name:
        return False
    
    # Check parameter list - function definitions should have type information
    paren_start = signature.find('(')
    paren_end = signature.rfind(')')
    if paren_start == -1 or paren_end == -1:
        return False
    
    params = signature[paren_start + 1:paren_end].strip()
    
    # Empty parameters are valid for function definitions
    if not params or params == 'void':
        return True
    
    # If parameters contain type keywords, likely a function definition
    if any(type_word in params.lower() for type_word in 
           ['int', 'char', 'void', 'struct', 'union', 'enum', 'const', 'unsigned', 'signed',
            'long', 'short', 'float', 'double', 'size_t', '*']):
        return True
    
    # If it's just simple variable names separated by commas, might be a function call
    simple_params = re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*(\s*,\s*[a-zA-Z_][a-zA-Z0-9_]*)*$', params)
    if simple_params:
        return False
    
    # Default to function definition if we can't determine otherwise
    return True
def is_function_signature(signature, all_lines, start_line):
    """Determine if a signature is actually a function definition"""
    
    # Must have parentheses
    if '(' not in signature or ')' not in signature:
        return False
    
    # Skip if it contains control flow keywords
    if any(keyword in signature.lower() for keyword in 
           ['if ', 'else', 'while ', 'for ', 'switch ', 'case ', 'default',
            'return ', 'break', 'continue', 'goto ', 'sizeof', 'typeof']):
        return False
    
    # Skip if it contains operators that suggest it's not a function definition
    if any(op in signature for op in ['==', '!=', '<=', '>=', '&&', '||', '!', '+', '-', '*/', '%']):
        return False
    
    # Skip assignments
    if '=' in signature.split('(')[0]:
        return False
    
    # Extract the part before parentheses
    before_paren = signature.split('(')[0].strip()
    if not before_paren:
        return False
    
    # Split into words and find the function name
    words = before_paren.split()
    if not words:
        return False
    
    # Find the last word that could be a function name
    function_name = None
    for word in reversed(words):
        # Clean the word of pointers
        clean_word = word.replace('*', '').strip()
        
        # Skip type specifiers and keywords
        if clean_word.lower() in ['static', 'inline', 'const', 'volatile', 'extern', 
                                 'struct', 'union', 'enum', 'unsigned', 'signed',
                                 'long', 'short', 'int', 'char', 'float', 'double', 
                                 'void', 'size_t', 'ssize_t', 'off_t', 'time_t', 
                                 'pid_t', 'uid_t', 'gid_t', 'dev_t', 'ino_t', 
                                 'mode_t', 'nlink_t', 'blkcnt_t', 'blksize_t',
                                 '__attribute__', '__restrict__', 'register',
                                 'auto', 'restrict', '__inline__']:
            continue
        
        # Must be a valid identifier
        if clean_word and clean_word.replace('_', '').isalnum() and not clean_word[0].isdigit():
            function_name = clean_word
            break
    
    # Must have found a valid function name
    if not function_name:
        return False
    
    # Check parameter list - function definitions should have type information
    paren_start = signature.find('(')
    paren_end = signature.rfind(')')
    if paren_start == -1 or paren_end == -1:
        return False
    
    params = signature[paren_start + 1:paren_end].strip()
    
    # Empty parameters are valid for function definitions
    if not params or params == 'void':
        return True
    
    # If parameters contain type keywords, likely a function definition
    if any(type_word in params.lower() for type_word in 
           ['int', 'char', 'void', 'struct', 'union', 'enum', 'const', 'unsigned', 'signed',
            'long', 'short', 'float', 'double', 'size_t', '*']):
        return True
    
    # If it's just simple variable names separated by commas, might be a function call
    simple_params = re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*(\s*,\s*[a-zA-Z_][a-zA-Z0-9_]*)*$', params)
    if simple_params:
        return False
    
    # Default to function definition if we can't determine otherwise
    return True
def is_function_definition(line, all_lines, line_index):
    """Determine if a line is actually a function definition"""
    
    # Skip obvious non-function patterns
    if any(keyword in line.lower() for keyword in 
           ['if ', 'else', 'while ', 'for ', 'switch ', 'do ', 
            'case ', 'default:', 'goto ', 'return ', 'break', 'continue']):
        return False
    
    # Skip assignments
    if '=' in line.split('(')[0]:
        return False
    
    # Skip if starts with closing brace
    if line.startswith('}'):
        return False
    
    # Must have parentheses
    if '(' not in line or ')' not in line:
        return False
    
    # Extract the part before parentheses
    before_paren = line.split('(')[0].strip()
    if not before_paren:
        return False
    
    # Split into words and find the function name
    words = before_paren.split()
    if not words:
        return False
    
    # Find the last word that could be a function name
    function_name = None
    for word in reversed(words):
        # Skip type specifiers and keywords
        if word.lower() in ['static', 'inline', 'const', 'volatile', 'extern', 
                           'struct', 'union', 'enum', 'unsigned', 'signed',
                           'long', 'short', 'int', 'char', 'float', 'double', 
                           'void', 'size_t', 'ssize_t', 'off_t', 'time_t', 
                           'pid_t', 'uid_t', 'gid_t', 'dev_t', 'ino_t', 
                           'mode_t', 'nlink_t', 'blkcnt_t', 'blksize_t',
                           '__attribute__', '__restrict__', 'register',
                           'auto', 'restrict']:
            continue
        
        # Skip if it ends with * (pointer type)
        if word.endswith('*'):
            continue
        
        # Skip if it's a type followed by *
        if word in ['struct', 'union', 'enum']:
            continue
        
        # Must be a valid identifier
        if word.replace('_', '').replace('*', '').isalnum() and len(word) > 0:
            function_name = word
            break
    
    # Must have found a valid function name
    if not function_name:
        return False
    
    # Additional validation: check if this looks like a function call vs definition
    # Function definitions typically have more complex parameter lists
    paren_content = line[line.find('(')+1:line.find(')')]
    
    # If it's just empty parentheses or simple parameters, might be a call
    if not paren_content.strip():
        # Empty parameters - could be function definition
        return True
    
    # Check for function call patterns (simple variable names)
    simple_params = re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*(\s*,\s*[a-zA-Z_][a-zA-Z0-9_]*)*$', paren_content.strip())
    if simple_params:
        # This looks like a function call, not definition
        return False
    
    # If we have complex parameters (types, pointers, etc.), likely a definition
    return True

def extract_function_name_from_signature(signature):
    """Extract the function name from a function signature"""
    before_paren = signature.split('(')[0].strip()
    words = before_paren.split()
    
    # Find the last word that could be a function name
    for word in reversed(words):
        clean_word = word.replace('*', '').strip()
        
        # Skip type specifiers and keywords
        if clean_word.lower() in ['static', 'inline', 'const', 'volatile', 'extern', 
                                 'struct', 'union', 'enum', 'unsigned', 'signed',
                                 'long', 'short', 'int', 'char', 'float', 'double', 
                                 'void', 'size_t', 'ssize_t', 'off_t', 'time_t', 
                                 'pid_t', 'uid_t', 'gid_t', 'dev_t', 'ino_t', 
                                 'mode_t', 'nlink_t', 'blkcnt_t', 'blksize_t',
                                 '__attribute__', '__restrict__', 'register',
                                 'auto', 'restrict', '__inline__']:
            continue
        
        if clean_word and clean_word.replace('_', '').isalnum() and not clean_word[0].isdigit():
            return clean_word
    
    return "unknown"

def count_lines(code):
    """Count the number of non-empty lines in the code"""
    if not code:
        return 0
    
    lines = code.split('\n')
    non_empty_lines = [line for line in lines if line.strip()]
    return len(non_empty_lines)

def update_database_metrics(conn, db_name):
    """Update metrics for a specific database"""
    print(f"\nUpdating metrics for {db_name}...")
    
    # Read the data
    df = pd.read_sql_query("SELECT id, VULNERABLE_CODE_BLOCK, PATCHED_CODE_BLOCK FROM vulnerabilities", conn)
    cursor = conn.cursor()
    
    for index, row in df.iterrows():
        print(f"Processing record {row['id']} from {db_name}...")
        
        # Calculate metrics
        num_files_changed = count_file_paths(row['PATCHED_CODE_BLOCK'])
        num_functions_changed = count_functions(row['PATCHED_CODE_BLOCK'])
        num_lines_vulnerable = count_lines(row['VULNERABLE_CODE_BLOCK'])
        num_lines_patched = count_lines(row['PATCHED_CODE_BLOCK'])
        
        # Log the calculated values
        print(f"  NUM_FILES_CHANGED: {num_files_changed}")
        print(f"  NUM_FUNCTIONS_CHANGED: {num_functions_changed}")
        print(f"  NUM_LINES_IN_VULNERABLE_CODE_BLOCK: {num_lines_vulnerable}")
        print(f"  NUM_LINES_IN_PATCHED_CODE_BLOCK: {num_lines_patched}")
        
        # Update the database
        cursor.execute("""
            UPDATE vulnerabilities 
            SET NUM_FILES_CHANGED = ?, 
                NUM_FUNCTIONS_CHANGED = ?, 
                NUM_LINES_IN_VULNERABLE_CODE_BLOCK = ?, 
                NUM_LINES_IN_PATCHED_CODE_BLOCK = ? 
            WHERE id = ?
        """, (num_files_changed, num_functions_changed, num_lines_vulnerable, num_lines_patched, row['id']))
    
    conn.commit()
    print(f"Metrics updated for {db_name}")

# Update both databases
update_database_metrics(conn1, "database.sqlite")
update_database_metrics(conn2, "database_leakagefree.sqlite")

# Close connections
conn1.close()
conn2.close()

print("\n\nAll metrics updated for both databases")


Updating metrics for database.sqlite...
Processing record 1 from database.sqlite...
  Found function definition ending at line 6: adjust_ptr_min_max_vals
  NUM_FILES_CHANGED: 1
  NUM_FUNCTIONS_CHANGED: 1
  NUM_LINES_IN_VULNERABLE_CODE_BLOCK: 175
  NUM_LINES_IN_PATCHED_CODE_BLOCK: 179
Processing record 2 from database.sqlite...
  Found function definition ending at line 3: dev_map_init_map
  NUM_FILES_CHANGED: 1
  NUM_FUNCTIONS_CHANGED: 1
  NUM_LINES_IN_VULNERABLE_CODE_BLOCK: 31
  NUM_LINES_IN_PATCHED_CODE_BLOCK: 29
Processing record 3 from database.sqlite...
  Found function definition ending at line 3: htab_map_alloc
  NUM_FILES_CHANGED: 1
  NUM_FUNCTIONS_CHANGED: 1
  NUM_LINES_IN_VULNERABLE_CODE_BLOCK: 102
  NUM_LINES_IN_PATCHED_CODE_BLOCK: 103
Processing record 4 from database.sqlite...
  Found function definition ending at line 3: stack_map_alloc
  NUM_FILES_CHANGED: 1
  NUM_FUNCTIONS_CHANGED: 1
  NUM_LINES_IN_VULNERABLE_CODE_BLOCK: 42
  NUM_LINES_IN_PATCHED_CODE_BLOCK: 42
Process

# fix database schema for VULNERABILITY_CWE

In [1]:
#!/usr/bin/env python3
import json
import os
import re
import sqlite3
from pathlib import Path
from typing import List

DBS = [
    "/home/azibaeir/Research/VulnLLMEval-SANER/data/database.sqlite",
    "/home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite",
]

CWE_RE = re.compile(r"CWE-\d{1,5}")

def parse_cwe_field(raw: str) -> List[str]:
    if raw is None:
        return []
    s = str(raw).strip()
    if not s:
        return []
    # Already JSON?
    try:
        val = json.loads(s)
        if isinstance(val, list):
            # normalize elements
            out = []
            for item in val:
                m = CWE_RE.search(str(item))
                if m:
                    out.append(m.group(0))
            return sorted(set(out))
        # if single string
        m = CWE_RE.search(str(val))
        return [m.group(0)] if m else []
    except Exception:
        pass
    # Handle formats like "[CWE-20, CWE-617]" or "CWE-20, CWE-617"
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]
    parts = re.split(r"[;,]\s*|\s+", s)
    out = []
    for p in parts:
        m = CWE_RE.search(p)
        if m:
            out.append(m.group(0))
    if out:
        return sorted(set(out))
    # Fallback: try to extract a single CWE number that might be like "20" or "20.0"
    num = None
    try:
        num = int(float(s))
    except Exception:
        pass
    return [f"CWE-{num}"] if num is not None else []

def ensure_link_table(cur):
    cur.execute("""
        CREATE TABLE IF NOT EXISTS vulnerability_cwes (
            commit_hash TEXT NOT NULL,
            cwe TEXT NOT NULL,
            PRIMARY KEY (commit_hash, cwe)
        )
    """)

def migrate_db(db_path: str):
    if not os.path.exists(db_path):
        print(f"Skip missing DB: {db_path}")
        return
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    ensure_link_table(cur)

    # Read all rows with commit and CWE field
    cur.execute("SELECT COMMIT_HASH, VULNERABILITY_CWE FROM vulnerabilities")
    rows = cur.fetchall()

    # Prepare updates
    updates = []
    link_rows = []
    for commit_hash, raw_cwe in rows:
        cwes = parse_cwe_field(raw_cwe)
        json_text = json.dumps(cwes, ensure_ascii=False)
        updates.append((json_text, commit_hash))
        for c in cwes:
            link_rows.append((commit_hash, c))

    # Update normalized JSON back into vulnerabilities
    cur.executemany(
        "UPDATE vulnerabilities SET VULNERABILITY_CWE = ? WHERE COMMIT_HASH = ?",
        updates,
    )

    # Populate link table
    if link_rows:
        cur.executemany(
            "INSERT OR IGNORE INTO vulnerability_cwes (commit_hash, cwe) VALUES (?, ?)",
            link_rows,
        )

    conn.commit()
    conn.close()
    print(f"Migrated {db_path}: {len(rows)} rows processed, {len(link_rows)} link rows created.")

def main():
    for db in DBS:
        migrate_db(db)

if __name__ == "__main__":
    main()

Migrated /home/azibaeir/Research/VulnLLMEval-SANER/data/database.sqlite: 316 rows processed, 316 link rows created.
Migrated /home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite: 103 rows processed, 81 link rows created.


# completing singleton CWEs

In [8]:
#!/usr/bin/env python3
import os
import sys
import json
from typing import Iterable, Any, List, Optional, Dict, Set, Tuple

# Target file (adjust if needed)
MEGAVUL_PATH = "/home/azibaeir/Research/megavul/megavul/c_cpp/megavul.json"
TARGET_CWE = "CWE-271"

def extract_cve(obj: Dict[str, Any]) -> Optional[str]:
    # Common field names for CVE id
    for key in ("cve_id", "cve", "CVE", "cveId", "cveID"):
        v = obj.get(key)
        if isinstance(v, str) and v.strip():
            return v.strip()
    # Sometimes nested
    nested = obj.get("metadata") or obj.get("vulnerability") or {}
    if isinstance(nested, dict):
        for key in ("cve_id", "cve"):
            v = nested.get(key)
            if isinstance(v, str) and v.strip():
                return v.strip()
    return None

def extract_cwe_ids(obj: Dict[str, Any]) -> List[str]:
    # Accept string or list
    cwe = obj.get("cwe_ids") or obj.get("cwe") or obj.get("cwes")
    out: List[str] = []
    if isinstance(cwe, str):
        # Normalize, split on common separators and brackets
        s = cwe.strip().strip("[](){}")
        parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]
        out = parts or ([s] if s else [])
    elif isinstance(cwe, list):
        out = [str(x).strip() for x in cwe if str(x).strip()]
    elif isinstance(cwe, dict):
        # Sometimes it’s dict-like: {"primary":"CWE-77", "others":[...]}
        for k in ("primary", "id", "value"):
            v = cwe.get(k)
            if isinstance(v, str) and v.strip():
                out.append(v.strip())
        for k in ("others", "ids", "values"):
            vals = cwe.get(k)
            if isinstance(vals, list):
                out.extend([str(x).strip() for x in vals if str(x).strip()])
    return list({x for x in out if x})

def extract_commit_hashes(obj: Dict[str, Any]) -> List[str]:
    candidates: List[str] = []

    def add_hash(h: Any):
        s = str(h).strip()
        if len(s) >= 7:  # basic sanity
            candidates.append(s)

    # Common fields
    for key in ("commit_hash", "commit", "fix_commit", "fix", "hash"):
        v = obj.get(key)
        if isinstance(v, str):
            add_hash(v)
        elif isinstance(v, list):
            for h in v:
                add_hash(h)

    # More common structures
    for key in ("commits", "commit_hashes", "fixes", "patches"):
        v = obj.get(key)
        if isinstance(v, list):
            for item in v:
                if isinstance(item, str):
                    add_hash(item)
                elif isinstance(item, dict):
                    for kk in ("hash", "commit", "commit_hash", "id"):
                        if kk in item:
                            add_hash(item[kk])

    # Nested containers
    for container_key in ("metadata", "fix", "repo", "source", "evidence"):
        nested = obj.get(container_key)
        if isinstance(nested, dict):
            for kk in ("commit", "commit_hash", "hash"):
                if kk in nested:
                    add_hash(nested[kk])
            for kk in ("commits", "commit_hashes", "fixes"):
                vv = nested.get(kk)
                if isinstance(vv, list):
                    for h in vv:
                        if isinstance(h, str):
                            add_hash(h)
                        elif isinstance(h, dict):
                            for k2 in ("hash", "commit", "commit_hash", "id"):
                                if k2 in h:
                                    add_hash(h[k2])

    # Deduplicate, preserve order
    seen = set()
    uniq = []
    for h in candidates:
        if h not in seen:
            seen.add(h)
            uniq.append(h)
    return uniq

def extract_repo_name(obj: Dict[str, Any]) -> Optional[str]:
    # Try common fields for repo name
    for key in ("repo_name", "repository", "repo", "project", "project_name"):
        v = obj.get(key)
        if isinstance(v, str) and v.strip():
            return v.strip()
        # Sometimes nested dict with "name" or "full_name"
        if isinstance(v, dict):
            for subkey in ("name", "full_name"):
                subv = v.get(subkey)
                if isinstance(subv, str) and subv.strip():
                    return subv.strip()
    # Try nested containers
    for container_key in ("metadata", "source", "vulnerability"):
        nested = obj.get(container_key)
        if isinstance(nested, dict):
            for key in ("repo_name", "repository", "repo", "project", "project_name"):
                v = nested.get(key)
                if isinstance(v, str) and v.strip():
                    return v.strip()
                if isinstance(v, dict):
                    for subkey in ("name", "full_name"):
                        subv = v.get(subkey)
                        if isinstance(subv, str) and subv.strip():
                            return subv.strip()
    return None

def iter_json_array(path: str) -> Iterable[Dict[str, Any]]:
    # Stream-read large JSON array without loading it entirely
    # Tries ijson first; if unavailable, falls back to manual streaming
    try:
        import ijson  # type: ignore
        with open(path, "rb") as f:
            for item in ijson.items(f, "item"):
                if isinstance(item, dict):
                    yield item
        return
    except Exception:
        pass

    # Fallback: naive streaming for a single top-level array
    decoder = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
    # Skip whitespace and opening '['
    i = 0
    while i < len(data) and data[i].isspace():
        i += 1
    if i < len(data) and data[i] == "[":
        i += 1
    # Parse elements separated by commas until ']'
    while i < len(data):
        while i < len(data) and data[i].isspace():
            i += 1
        if i < len(data) and data[i] == "]":
            break
        obj, end = decoder.raw_decode(data, idx=i)
        if isinstance(obj, dict):
            yield obj
        i = end
        while i < len(data) and data[i].isspace():
            i += 1
        if i < len(data) and data[i] == ",":
            i += 1

def main():
    path = MEGAVUL_PATH
    if not os.path.exists(path):
        print(f"File not found: {path}", file=sys.stderr)
        sys.exit(1)

    target = TARGET_CWE
    seen: Set[Tuple[str, str, str]] = set()
    out_lines = []
    for obj in iter_json_array(path):
        cwes = extract_cwe_ids(obj)
        if target in cwes:
            cve = extract_cve(obj) or "UNKNOWN_CVE"
            repo_name = extract_repo_name(obj) or "UNKNOWN_REPO"
            commits = extract_commit_hashes(obj) or ["UNKNOWN_COMMIT"]
            for h in commits:
                key = (cve, h, repo_name)
                if key not in seen:
                    seen.add(key)
                    out_lines.append(f"{cve}\t{h}\t{repo_name}")

    # Print results (CVE<TAB>commit_hash<TAB>repo_name), no duplicates
    for line in out_lines:
        print(line)

if __name__ == "__main__":
    main()

# CWE distribution for leakage free database

In [9]:
#!/usr/bin/env python3
import os
import json
import sqlite3
from collections import Counter, defaultdict

DB = "/home/azibaeir/Research/VulnLLMEval-SANER/data/database_leakagefree.sqlite"

def get_conn(db_path: str):
    if not os.path.exists(db_path):
        raise FileNotFoundError(db_path)
    return sqlite3.connect(db_path)

def load_cwes_from_link_table(cur):
    cur.execute("SELECT cwe, COUNT(*) FROM vulnerability_cwes GROUP BY cwe ORDER BY COUNT(*) DESC")
    per_cwe = Counter({cwe: n for cwe, n in cur.fetchall()})
    cur.execute("SELECT commit_hash, cwe FROM vulnerability_cwes")
    per_sample = defaultdict(list)
    for commit_hash, cwe in cur.fetchall():
        per_sample[commit_hash].append(cwe)
    return per_cwe, per_sample

def load_cwes_from_json(cur):
    cur.execute("SELECT COMMIT_HASH, VULNERABILITY_CWE FROM vulnerabilities")
    per_cwe = Counter()
    per_sample = defaultdict(list)
    for commit_hash, raw in cur.fetchall():
        if not raw:
            continue
        try:
            arr = json.loads(raw)
            if isinstance(arr, list):
                cwes = [str(x).strip() for x in arr if str(x).strip()]
            else:
                cwes = [str(arr).strip()]
        except Exception:
            # fallback if stored as plain string like "CWE-77, CWE-79"
            s = str(raw).strip().strip("[]")
            cwes = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]
        for c in cwes:
            per_cwe[c] += 1
            per_sample[commit_hash].append(c)
    return per_cwe, per_sample

def main():
    conn = get_conn(DB)
    cur = conn.cursor()

    # Prefer link table if present
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='vulnerability_cwes'")
    has_link = cur.fetchone() is not None

    per_cwe, per_sample = load_cwes_from_link_table(cur) if has_link else load_cwes_from_json(cur)

    # Summary
    unique_cwes = len(per_cwe)
    total_links = sum(per_cwe.values())
    print(f"Unique CWEs: {unique_cwes}")
    print(f"Total commit–CWE links: {total_links}")

    # Counts per CWE (desc)
    print("\nCWE counts:")
    for cwe, n in per_cwe.most_common():
        print(f"{cwe}\t{n}")

    # Per-sample listing (commit -> CWEs)
    print("\nPer-sample CWEs (commit_hash<TAB>comma_separated_CWEs):")
    for commit_hash, cwes in per_sample.items():
        print(f"{commit_hash}\t{','.join(sorted(set(cwes)))}")

    conn.close()

if __name__ == "__main__":
    main()

Unique CWEs: 18
Total commit–CWE links: 104

CWE counts:
CWE-416	35
CWE-125	14
CWE-476	13
CWE-787	6
CWE-667	6
CWE-190	5
CWE-129	5
CWE-20	4
CWE-908	3
CWE-835	3
CWE-401	3
CWE-770	1
CWE-682	1
CWE-662	1
CWE-415	1
CWE-362	1
CWE-193	1
CWE-120	1

Per-sample CWEs (commit_hash<TAB>comma_separated_CWEs):
77de19b6867f2740cdcb6c9c7e50d522b47847a4	CWE-129
ee735aa33db16c1fb5ebccbaf84ad38f5583f3cc	CWE-129
d8df010f72b8a32aaea393e36121738bb53ed905	CWE-476
f2176a07e7b19f73e05c805cf3d130a2999154cb	CWE-476
fdf480da5837c23b146c4743c18de97202fcab37	CWE-125
3b32b7f638fe61e9d29290960172f4e360e38233	CWE-908
107a23185d990e3df6638d9a84c835f963fe30a6	CWE-125
c1baf6528bcfd6a86842093ff3f8ff8caf309c12	CWE-125
d19d7345a7bcdb083b65568a11b11adffe0687af	CWE-125
76e51db43fe4aaaebcc5ddda67b0807f7c9bdecc	CWE-129
647cef20e649c576dff271e018d5d15d998b629d	CWE-667
2fc9feff45d92a92cd5f96487655d5be23fb7e2b	CWE-416
75845c6c1a64483e9985302793dbf0dfa5f71e32	CWE-416
56d5f3eba3f5de0efdd556de4ef381e109b973a9	CWE-476
b2df03ed4052e97126

In [11]:
#!/usr/bin/env python3
import sqlite3
import json
import os
from typing import Set

DB = "/home/azibaeir/Research/VulnLLMEval-SANER/data/database.sqlite"

TARGET_CWES = {
    "CWE-416","CWE-125","CWE-476","CWE-787","CWE-667","CWE-190","CWE-129","CWE-20",
    "CWE-908","CWE-835","CWE-401","CWE-770","CWE-682","CWE-662","CWE-415","CWE-362",
    "CWE-193","CWE-120"
}

def get_present_cwes(conn: sqlite3.Connection) -> Set[str]:
    cur = conn.cursor()
    # Prefer normalized link table if present
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='vulnerability_cwes'")
    if cur.fetchone():
        cur.execute("SELECT DISTINCT cwe FROM vulnerability_cwes")
        return {row[0].strip() for row in cur.fetchall() if row and row[0]}

    # Fallback: parse JSON/text from vulnerabilities.VULNERABILITY_CWE
    cur.execute("SELECT VULNERABILITY_CWE FROM vulnerabilities WHERE VULNERABILITY_CWE IS NOT NULL AND TRIM(VULNERABILITY_CWE)!=''")
    present = set()
    for (raw,) in cur.fetchall():
        s = str(raw).strip()
        # Try JSON array or string first
        try:
            val = json.loads(s)
            if isinstance(val, list):
                present.update({str(x).strip() for x in val if str(x).strip()})
            elif isinstance(val, str):
                val = val.strip()
                if val:
                    present.add(val)
            else:
                # If some other JSON, fall back to splitting
                raise ValueError
        except Exception:
            s = s.strip("[]")
            for part in s.replace(";", ",").split(","):
                p = part.strip()
                if p:
                    present.add(p)
    return present

def main():
    if not os.path.exists(DB):
        raise FileNotFoundError(DB)
    with sqlite3.connect(DB) as conn:
        present = get_present_cwes(conn)

    missing = sorted(TARGET_CWES - present)
    if missing:
        print("Missing:", ",".join(missing))
    else:
        print("All present")

if __name__ == "__main__":
    main()

All present
